# LangSmith Evaluation

Offline evaluation of the AKS chatbot agent using [LangSmith Evaluation](https://docs.smith.langchain.com/evaluation).

### What this notebook does
1. Defines a **golden dataset** of Q&A pairs with expected answers
2. Wraps the agent as a `@traceable` **target function**
3. Defines three **evaluators** (LLM-as-judge):
   - `correctness` — is the answer factually correct given a reference?
   - `relevance` — does the answer address the question (no reference needed)?
   - `groundedness` — is the answer grounded in the retrieved context?
4. Runs `client.evaluate()` and prints a summary table
5. Compares against a **baseline** experiment to detect regressions

**Pre-requisites**: `LANGSMITH_API_KEY`, `OPENAI_API_KEY` in `.env`

In [ ]:
# ── 1. Environment setup ──────────────────────────────────────────────────────
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env", override=False)

LANGSMITH_API_KEY = os.environ["LANGSMITH_API_KEY"]
OPENAI_API_KEY    = os.environ["OPENAI_API_KEY"]
LANGSMITH_PROJECT = os.environ.get("LANGSMITH_PROJECT", "aks-chatbot")

os.environ.setdefault("LANGSMITH_TRACING", "true")

print(f"Project: {LANGSMITH_PROJECT}")

In [ ]:
# ── 2. Golden dataset ─────────────────────────────────────────────────────────
# Mirrors evaluation/golden_dataset.jsonl — edit there to keep them in sync.
from langsmith import Client

client = Client(api_key=LANGSMITH_API_KEY)

DATASET_NAME = "aks-chatbot-golden-v1"

golden_examples = [
    {
        "input":    "What is Kubernetes?",
        "expected": "Kubernetes is an open-source container orchestration platform that automates deployment, scaling, and management of containerised applications.",
    },
    {
        "input":    "What does AKS stand for and what cloud platform does it run on?",
        "expected": "AKS stands for Azure Kubernetes Service and it runs on Microsoft Azure.",
    },
    {
        "input":    "How does a Helm chart help with Kubernetes deployments?",
        "expected": "A Helm chart packages all Kubernetes manifests and configuration values into a reusable, versioned bundle that can be installed or upgraded with a single command.",
    },
    {
        "input":    "What is the capital of France?",
        "expected": "The capital of France is Paris.",
    },
    {
        "input":    "Explain what a PersistentVolumeClaim is in Kubernetes.",
        "expected": "A PersistentVolumeClaim (PVC) is a request for storage by a user that binds to a PersistentVolume (PV), abstracting the underlying storage details from the application.",
    },
]

# Create or reuse the dataset
existing = [d.name for d in client.list_datasets()]
if DATASET_NAME not in existing:
    dataset = client.create_dataset(DATASET_NAME, description="Golden Q&A pairs for the AKS chatbot agent")
    client.create_examples(
        inputs=[{"question": ex["input"]} for ex in golden_examples],
        outputs=[{"answer": ex["expected"]} for ex in golden_examples],
        dataset_id=dataset.id,
    )
    print(f"Dataset '{DATASET_NAME}' created with {len(golden_examples)} examples.")
else:
    print(f"Dataset '{DATASET_NAME}' already exists — reusing.")

In [ ]:
# ── 3. Target function ────────────────────────────────────────────────────────
# This function is called once per dataset example.
# It hits the live API so the run is traced in LangSmith automatically.

import requests
from langsmith import traceable

# Point to your running backend (local or port-forwarded k8s)
BASE_URL = os.environ.get("CHATBOT_API_URL", "http://localhost:8000")
EVAL_AUTH_TOKEN = os.environ.get("CHATBOT_EVAL_TOKEN", "")  # JWT for evaluation user

@traceable(name="chatbot_predict", project_name=LANGSMITH_PROJECT)
def predict(inputs: dict) -> dict:
    """Call the chatbot API and return {answer, context}."""
    question = inputs["question"]

    # If no live API is available, fall back to calling the agent directly
    if not EVAL_AUTH_TOKEN:
        from langchain_openai import ChatOpenAI
        llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY)
        response = llm.invoke(question)
        return {"answer": response.content, "context": ""}

    headers = {"Authorization": f"Bearer {EVAL_AUTH_TOKEN}"}
    # Create a new conversation
    conv = requests.post(f"{BASE_URL}/api/v1/conversations", headers=headers, timeout=30)
    conv.raise_for_status()
    thread_id = conv.json()["id"]

    # Send the question and collect the streamed response
    resp = requests.post(
        f"{BASE_URL}/api/v1/conversations/{thread_id}/messages",
        json={"message": question},
        headers=headers,
        stream=True,
        timeout=60,
    )
    resp.raise_for_status()

    import json as _json
    answer_parts = []
    for line in resp.iter_lines():
        if not line:
            continue
        text = line.decode("utf-8").removeprefix("data: ")
        try:
            event = _json.loads(text)
            if event.get("type") == "chunk":
                answer_parts.append(event["content"])
        except Exception:
            pass

    return {"answer": "".join(answer_parts), "context": ""}

In [ ]:
# ── 4. Evaluators (LLM-as-judge) ──────────────────────────────────────────────
from typing import TypedDict
from langchain_openai import ChatOpenAI

judge_llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0)


class CorrectnessGrade(TypedDict):
    score: int    # 0 or 1
    reason: str


class RelevanceGrade(TypedDict):
    score: int
    reason: str


class GroundednessGrade(TypedDict):
    score: int
    reason: str


_correctness_llm  = judge_llm.with_structured_output(CorrectnessGrade)
_relevance_llm    = judge_llm.with_structured_output(RelevanceGrade)
_groundedness_llm = judge_llm.with_structured_output(GroundednessGrade)

CORRECTNESS_PROMPT = """\
You are a strict evaluator. Given a question, a reference answer, and a candidate answer,
decide whether the candidate answer is factually correct and complete relative to the reference.
Score 1 if correct, 0 if incorrect or missing key facts.

Question: {question}
Reference: {reference}
Candidate: {answer}"""

RELEVANCE_PROMPT = """\
You are a strict evaluator. Given a question and an answer, decide whether the answer
directly addresses the question without going off-topic.
Score 1 if relevant, 0 if not relevant or completely off-topic.

Question: {question}
Answer:   {answer}"""

GROUNDEDNESS_PROMPT = """\
You are a strict evaluator. Given a question, an answer, and retrieved context,
decide whether every claim in the answer is supported by the context (or by common knowledge
when context is empty). Score 1 if grounded, 0 if the answer contains unsupported claims.

Question: {question}
Context:  {context}
Answer:   {answer}"""


def correctness_evaluator(run, example):
    """LangSmith evaluator: factual correctness vs. reference."""
    question  = example.inputs["question"]
    reference = example.outputs["answer"]
    answer    = run.outputs["answer"]
    grade = _correctness_llm.invoke(
        CORRECTNESS_PROMPT.format(question=question, reference=reference, answer=answer)
    )
    return {"key": "correctness", "score": grade["score"], "comment": grade["reason"]}


def relevance_evaluator(run, example):
    """LangSmith evaluator: answer relevance (no reference required)."""
    question = example.inputs["question"]
    answer   = run.outputs["answer"]
    grade = _relevance_llm.invoke(
        RELEVANCE_PROMPT.format(question=question, answer=answer)
    )
    return {"key": "relevance", "score": grade["score"], "comment": grade["reason"]}


def groundedness_evaluator(run, example):
    """LangSmith evaluator: answer is grounded in retrieved context."""
    question = example.inputs["question"]
    answer   = run.outputs["answer"]
    context  = run.outputs.get("context", "")
    grade = _groundedness_llm.invoke(
        GROUNDEDNESS_PROMPT.format(question=question, context=context, answer=answer)
    )
    return {"key": "groundedness", "score": grade["score"], "comment": grade["reason"]}

In [ ]:
# ── 5. Run evaluation ─────────────────────────────────────────────────────────
import uuid

experiment_name = f"eval-{uuid.uuid4().hex[:8]}"

results = client.evaluate(
    predict,
    data=DATASET_NAME,
    evaluators=[correctness_evaluator, relevance_evaluator, groundedness_evaluator],
    experiment_prefix=experiment_name,
    num_repetitions=1,
    max_concurrency=2,
)

print(f"\nExperiment: {experiment_name}")
print(results)

In [ ]:
# ── 6. Summary table ──────────────────────────────────────────────────────────
import statistics

metrics = {"correctness": [], "relevance": [], "groundedness": []}

for result in results._results:
    for fb in result.get("evaluation_results", {}).get("results", []):
        key = fb.key
        if key in metrics and fb.score is not None:
            metrics[key].append(fb.score)

print(f"{'Metric':<16} {'Mean':>6}  {'Pass rate':>10}")
print("-" * 36)
for metric, scores in metrics.items():
    if scores:
        mean = statistics.mean(scores)
        pass_rate = sum(1 for s in scores if s >= 0.5) / len(scores)
        print(f"{metric:<16} {mean:>6.3f}  {pass_rate:>9.0%}")
    else:
        print(f"{metric:<16}   n/a")